# Results 2 — Selective pressures, concordant variant effects and directionality

Numbers for this Results subsection and the four data files behind Figure 2 and Extended
Data Figures 4 and 9.

| file | panel |
| --- | --- |
| `figure_2_a.csv` | mean absolute effect size across MAF bins, four study types (Extended Data Fig. 9) |
| `figure_2_b.csv` | proportion of protein-altering variants across MAF bins (Figure 2a) |
| `figure_2_c.csv` | consequence distribution per study type (Figure 2c) |
| `figure_2_d.csv` | mean absolute effect size per consequence, disease and measurement GWAS (Figure 2b) |
| `extended_figure_4.csv` | the same for eQTLs and cis-pQTLs (Extended Data Fig. 4) |

In [1]:
from gentropy.common.session import Session
from pyspark.sql import Window
from pyspark.sql import functions as f
from pyspark.sql import types as t

from manuscript_methods import paper
from manuscript_methods.consequence import ConsequenceCategory
from manuscript_methods.datasets import LeadVariantEffect
from manuscript_methods.maf import prepare_maf_bins
from manuscript_methods.ve import SingleVariantEffectMethod, VariantEffect

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

PAV_THRESHOLD = 0.66
MAF_SPLITS = [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 00:50:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
lve = LeadVariantEffect.from_parquet(session, paper.derived("replicated_lead_variants"))
print("replicated credible sets across the four study types:", lve.df.count())

replicated credible sets across the four study types: 459516


## One row per variant and trait

A variant can appear in several studies of the same trait with different effect sizes. Two
representatives are marked per variant and partition: the largest absolute effect
(`hasLargestAbsBeta`, used for effect sizes) and the most severe consequence
(`hasLargestVariantEffect`, used for the PAV proportion).

In [3]:
disease_ids = f.concat_ws(",", f.sort_array(f.array_distinct("diseaseIds")))
partition = f.concat_ws(",", f.coalesce(f.col("geneId"), disease_ids), f.col("studyType"))
by_beta = Window.partitionBy("variantId", "partition").orderBy(f.desc("absEstimatedBeta"))
by_consequence = Window.partitionBy("variantId", "partition").orderBy(f.desc("consequenceScore"))

effects = (
    lve.df.select(
        "variantId",
        "geneId",
        "diseaseIds",
        VariantEffect(f.col("variantEffect"))
        .filter_effect_by_method(SingleVariantEffectMethod.VEP)
        .normalised_score.alias("vepScore"),
        f.col("majorLdPopulationMaf.value").alias("maf"),
        f.col("rescaledStatistics.absEstimatedBeta").alias("absEstimatedBeta"),
        f.col("studyStatistics.studyType").alias("studyType"),
        f.col("leadVariantConsequence.mostSevereConsequence.transcriptConsequence.consequenceScore").alias(
            "consequenceScore"
        ),
        partition.alias("partition"),
    )
    # Deep intergenic variants have no transcript consequence; fall back to the variant-level score.
    .withColumn("consequenceScore", f.round(f.coalesce("consequenceScore", "vepScore"), 3))
    .withColumn("isProteinAltering", (f.col("consequenceScore") >= PAV_THRESHOLD).cast(t.ShortType()))
    .withColumn("hasLargestAbsBeta", f.row_number().over(by_beta) == 1)
    .withColumn("hasLargestVariantEffect", f.row_number().over(by_consequence) == 1)
    .drop("vepScore")
    .dropDuplicates(["variantId", "partition"])
    .cache()
)
# "Non-redundant" is one row per variant and trait, which is what the manuscript counts.
numbers["R2.01"] = effects.count()
print("non-redundant replicated credible sets with PIP >= 0.5:", numbers["R2.01"])

non-redundant replicated credible sets with PIP >= 0.5: 121490


## Figure 2a — proportion of protein-altering variants across MAF bins

In [4]:
per_bin = Window.partitionBy("studyType").orderBy("mafBin.mafBinIndex")
per_type = Window.partitionBy("studyType")

total = f.count(f.lit(1)).over(per_bin)
altering = f.sum("isProteinAltering").over(per_bin)
proportion = f.avg("isProteinAltering").over(per_bin)
stderr = f.sqrt(proportion * (1 - proportion) / total)
label = f.concat_ws(
    " ", f.col("studyType"), f.concat(f.lit("N=("), f.format_number(f.count(f.lit(1)).over(per_type), 0), f.lit(")"))
)

pav_by_maf = (
    prepare_maf_bins(effects.filter(f.col("hasLargestVariantEffect")), splits=MAF_SPLITS)
    .withColumn("totals_label", label)
    .withColumn(
        "PAV",
        f.struct(
            altering.alias("nAlteringInBucket"),
            (total - altering).alias("nNonAlteringInBucket"),
            total.alias("nTotalInBucket"),
            proportion.alias("alteringProportionInBucket"),
            stderr.alias("alteringProportionInBucketStderr"),
            (proportion - 1.96 * stderr).alias("alteringProportionInBucketCILower"),
            (proportion + 1.96 * stderr).alias("alteringProportionInBucketCIUpper"),
        ),
    )
    .drop("absEstimatedBeta", "maf", "isProteinAltering")
    .dropDuplicates()
    .select("PAV.*", "studyType", "mafBin.*", "totals_label")
    .orderBy("studyType", "mafBinIndex")
)
pav_by_maf.toPandas().to_csv(paper.derived("figure_2_b.csv"), index=False)
pav_by_maf.toPandas().head()

Using following buckets [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]


,nAlteringInBucket,nNonAlteringInBucket,nTotalInBucket,alteringProportionInBucket,alteringProportionInBucketStderr,alteringProportionInBucketCILower,alteringProportionInBucketCIUpper,studyType,mafBinRange,mafBinLower,mafBinUpper,mafBinMidpoint,mafBinIndex,totals_label
0,7,9,16,0.437500,0.124020,0.194422,0.680578,cis-pqtl,0.0-0.01,0.00,0.01,0.005,0,"cis-pqtl N=(1,270)"
1,64,178,242,0.264463,0.028352,0.208894,0.320032,cis-pqtl,0.01-0.05,0.01,0.05,0.030,1,"cis-pqtl N=(1,270)"
2,97,296,393,0.246819,0.021749,0.204191,0.289448,cis-pqtl,0.05-0.1,0.05,0.10,0.075,2,"cis-pqtl N=(1,270)"
3,134,491,625,0.214400,0.016416,0.182224,0.246576,cis-pqtl,0.1-0.2,0.10,0.20,0.150,3,"cis-pqtl N=(1,270)"
4,165,665,830,0.198795,0.013853,0.171644,0.225947,cis-pqtl,0.2-0.3,0.20,0.30,0.250,4,"cis-pqtl N=(1,270)"


## Extended Data Figure 9 — mean absolute effect size across MAF bins

In [5]:
per_bin = Window.partitionBy("studyType", "mafBin.mafBinIndex")
beta = f.col("absEstimatedBeta")
n = f.count(f.lit(1)).over(per_bin)
mean_beta = f.avg(beta).over(per_bin)
std_beta = f.std(beta).over(per_bin)
stderr = std_beta / f.sqrt(n)
label = f.concat_ws(
    " ", f.col("studyType"), f.concat(f.lit("N=("), f.format_number(f.count(f.lit(1)).over(per_type), 0), f.lit(")"))
)

beta_by_maf = (
    prepare_maf_bins(effects.filter(f.col("hasLargestAbsBeta")).filter(beta < 3), splits=MAF_SPLITS)
    .withColumn("totals_label", label)
    .withColumn(
        "absBetaOverMafBin",
        f.struct(
            f.min(beta).over(per_bin).alias("minAbsEstimatedBetaInBucker"),
            f.max(beta).over(per_bin).alias("maxAbsEstimatedBetaInBucker"),
            std_beta.alias("stdAbsEstimatedBetaInBucker"),
            n.alias("nVariantsInBucket"),
            mean_beta.alias("avgAbsEstimatedBetaInBucket"),
            stderr.alias("avgAbsEstimatedBetaInBucketStderr"),
            (mean_beta - 1.96 * stderr).alias("avgAbsEstimatedBetaInBucketCILower"),
            (mean_beta + 1.96 * stderr).alias("avgAbsEstimatedBetaInBucketCIUpper"),
        ),
    )
    .drop("absEstimatedBeta", "maf", "isProteinAltering")
    .dropDuplicates()
    .select("absBetaOverMafBin.*", "studyType", "mafBin.*", "totals_label")
    .orderBy("studyType", "mafBinIndex")
)
beta_by_maf.toPandas().to_csv(paper.derived("figure_2_a.csv"), index=False)
beta_by_maf.toPandas().head()

Using following buckets [0.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]


,minAbsEstimatedBetaInBucker,maxAbsEstimatedBetaInBucker,stdAbsEstimatedBetaInBucker,nVariantsInBucket,avgAbsEstimatedBetaInBucket,avgAbsEstimatedBetaInBucketStderr,avgAbsEstimatedBetaInBucketCILower,avgAbsEstimatedBetaInBucketCIUpper,studyType,mafBinRange,mafBinLower,mafBinUpper,mafBinMidpoint,mafBinIndex,totals_label
0,0.361968,2.028047,0.526436,16,1.015569,0.131609,0.757616,1.273523,cis-pqtl,0.0-0.01,0.00,0.01,0.005,0,"cis-pqtl N=(1,270)"
1,0.110754,1.489421,0.386249,226,0.669968,0.025693,0.619610,0.720326,cis-pqtl,0.01-0.05,0.01,0.05,0.030,1,"cis-pqtl N=(1,270)"
2,0.073253,2.091797,0.269561,151,0.435050,0.021937,0.392054,0.478046,cis-pqtl,0.05-0.1,0.05,0.10,0.075,2,"cis-pqtl N=(1,270)"
3,0.073263,1.504885,0.264808,232,0.406907,0.017385,0.372832,0.440983,cis-pqtl,0.1-0.2,0.10,0.20,0.150,3,"cis-pqtl N=(1,270)"
4,0.041626,1.117937,0.223829,205,0.332111,0.015633,0.301470,0.362751,cis-pqtl,0.2-0.3,0.20,0.30,0.250,4,"cis-pqtl N=(1,270)"


## Figure 2c — consequence distribution per study type

The exploded consequence table is ranked by severity per variant, study type and partition,
then deduplicated so each variant contributes once per consequence category.

In [6]:
consequences = session.spark.read.parquet(paper.derived("variant_consequences"))
ranked_window = Window.partitionBy("variantId", "studyType", "partition").orderBy(f.asc("ranking"))

ranked = (
    consequences.withColumn("ranking", ConsequenceCategory.ranking(f.col("consequenceCategory")))
    .withColumn("lowestInRanking", f.dense_rank().over(ranked_window))
    .withColumn("maxAbsEstimatedBeta", f.max("maxAbsEstimatedBeta").over(ranked_window))
    .withColumn("con", f.size(f.collect_list("consequenceCategory").over(ranked_window)))
    .dropDuplicates(["variantId", "consequenceCategory", "partition"])
    .cache()
)
print("ranked consequence rows:", ranked.count())

ranked consequence rows: 173195


In [7]:
per_type_consequence = Window.partitionBy("studyType", "consequenceCategory")
per_study_type = Window.partitionBy("studyType")

distribution = (
    ranked.withColumn("nTotal", f.count("variantId").over(per_study_type))
    .withColumn("nConsequence", f.count("variantId").over(per_type_consequence))
    .withColumn("pConsequenceValue", 100 * f.col("nConsequence") / f.col("nTotal"))
    .withColumn("pConsequneceLabel", f.concat(f.format_number("pConsequenceValue", 2), f.lit("%")))
    .drop("lowestInRanking", "variantId", "maxAbsEstimatedBeta")
    .dropDuplicates(["studyType", "consequenceCategory"])
    .orderBy(f.desc("nTotal"), "studyType", f.desc("nConsequence"))
    .toPandas()
)
distribution.to_csv(paper.derived("figure_2_c.csv"), index=False)

shares = distribution.pivot_table(index="consequenceCategory", columns="studyType", values="pConsequenceValue")
numbers["R2.02"] = round(float(shares.loc["enhancer", "gwas-disease"]), 1)
numbers["R2.03"] = round(float(shares.loc["promoter", "cis-pqtl"]), 1)
numbers["R2.04"] = round(float(shares.loc["promoter", "gwas-disease"]), 1)
numbers["R2.05"] = round(float(shares.loc[["intragenic", "protein_altering"], "gwas-disease"].sum()))
shares.round(2)

studyType,cis-pqtl,eqtl,gwas-disease,gwas-measurement
consequenceCategory,,,,
enhancer,13.18,10.23,13.80,12.68
intergenic,25.26,34.09,19.83,21.22
intragenic,41.33,49.52,52.19,52.37
promoter,9.09,4.74,2.87,3.51
protein_altering,11.13,1.42,11.30,10.21


## Figure 2b and Extended Data Figure 4 — effect size per consequence

In [8]:
def beta_by_consequence(study_types):
    """Mean absolute effect size and 95% CI per consequence category, for the given study types."""
    window = Window.partitionBy("studyType", "consequenceCategory")
    n = f.count("variantId").over(window)
    mean_beta = f.avg("maxAbsEstimatedBeta").over(window)
    std_beta = f.stddev("maxAbsEstimatedBeta").over(window)
    stderr = std_beta / f.sqrt(n)
    return (
        ranked.filter(f.col("studyType").isin(*study_types))
        .withColumn("nTotal", n)
        .withColumn("nConsequence", n)
        .withColumn("pConsequenceValue", f.lit(100.0))
        .withColumn("pConsequneceLabel", f.lit("100.00%"))
        .withColumn("avgMaxAbsEstimatedBeta", mean_beta)
        .withColumn("stdMaxAbsEstimatedBeta", std_beta)
        .withColumn("seMaxAbsEstimatedBeta", stderr)
        .withColumn("CILower", mean_beta - 1.96 * stderr)
        .withColumn("CIUpper", mean_beta + 1.96 * stderr)
        .withColumn(
            "statisticLabel",
            f.concat_ws(
                " / ",
                f.col("consequenceCategory"),
                f.concat_ws("=", f.lit("N"), f.col("nTotal")),
                f.concat_ws(
                    "\u00b1",
                    f.concat_ws("=", f.lit("\u03b2\u0302"), f.round("avgMaxAbsEstimatedBeta", 3)),
                    f.round(1.96 * f.col("seMaxAbsEstimatedBeta"), 3),
                ),
            ),
        )
        .drop("lowestInRanking", "variantId", "maxAbsEstimatedBeta")
        .dropDuplicates(["studyType", "consequenceCategory"])
        .toPandas()
    )


gwas = beta_by_consequence(["gwas-disease", "gwas-measurement"])
gwas.to_csv(paper.derived("figure_2_d.csv"), index=False)

molqtl = beta_by_consequence(["eqtl", "cis-pqtl"])
molqtl.to_csv(paper.derived("extended_figure_4.csv"), index=False)

gwas.pivot_table(index="consequenceCategory", columns="studyType", values="avgMaxAbsEstimatedBeta").round(3)

26/08/19 00:51:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


studyType,gwas-disease,gwas-measurement
consequenceCategory,,
enhancer,0.218,0.095
intergenic,0.205,0.094
intragenic,0.205,0.094
promoter,0.254,0.171
protein_altering,0.356,0.212


## Numbers

In [9]:
import pandas as pd

print(paper.save_results("selective_pressures", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/selective_pressures.json


,computed
R2.01,121490.0
R2.02,13.8
R2.03,9.1
R2.04,2.9
R2.05,63.0
